# Profiling notebook for generic volume data

In [1]:
%load_ext line_profiler
%load_ext autoreload
%autoreload 2

Failed to read module file 'c:\Users\mariu\AppData\Local\Programs\Python\Python312\Lib\urllib\parse.py' for module 'urllib.parse': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\mariu\AppData\Roaming\Python\Python312\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mariu\AppData\Roaming\Python\Python312\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\mariu\AppData\Local\Programs\Python\Python312\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen import

In [ ]:
import timeit
import numpy as np
from PIL import Image
import numpy.lib.recfunctions as rf
import os
os
from _preprocess_module import HeightFieldExtractor
from smooth_generic_volume_container import SmoothGenericVolumeContainer
from generic_volume_container import GenericVolumeContainer
import create_volume

ImportError: DLL load failed while importing _preprocess_module: Das angegebene Modul wurde nicht gefunden.

In [ ]:
VOLUME = create_volume.create_huge_volume()

In [ ]:
def main():

    preprocessor = HeightFieldExtractor((create_volume.RESOLUTION_Y, create_volume.RESOLUTION_X), 2, 256)
    preprocessor.add_volume(VOLUME.container, (create_volume.RESOLUTION_X, create_volume.RESOLUTION_Y, create_volume.RESOLUTION_Z), 0.001)

    extended_heightfield, normal_map = preprocessor.extract_data_representation( 0.0 )

    # Save the extended heightfields
    for z in range(extended_heightfield.shape[2]):
        # entry_0 = rf.structured_to_unstructured(extended_heightfield[:,:,z]);
        entry = extended_heightfield[:,:,z]
        entry = entry.astype(np.uint16)
        img = Image.fromarray(entry, "I;16")
        img.save("output/integrated_"+str(z)+".tif")

    # Convert the first normal map
    normal_map = normal_map.squeeze(2)
    normal_map = rf.structured_to_unstructured( normal_map )
    normal_map = ( normal_map + 1.0 ) * 127.5
    normal = normal_map.astype(np.uint8)
    # print(normal)
    img = Image.fromarray(normal, "RGB")
    img.save("output/normal.tif")
    del preprocessor

In [ ]:
# for _ in range(3):
#     main()
# %timeit main()
%lprun -f main main()